# Azul Rocket — run everything from the GitHub repo

[github.com/laAzulNanotec/azul-rocket](https://github.com/laAzulNanotec/azul-rocket)

Clones the public repo into Colab on every run, then generates plots into a separate output directory.

**Directories:**
- `/content/azul-rocket/` — fresh clone of the repo (read-only, treat as immutable)
- `/content/output/` — your generated PNGs and any custom analysis (this is what you download)

**Just run the cells in order.**

## Setup — fresh clone of the repo

This cell deletes any existing local clone and pulls fresh from GitHub. That way you always get the latest commits and there's never a merge conflict from local changes.

In [ ]:
# Always start with a clean clone of the repo
import os, shutil

REPO_URL = 'https://github.com/laAzulNanotec/azul-rocket.git'
REPO_DIR = '/content/azul-rocket'
OUT_DIR  = '/content/output'

# CRITICAL: cd to /content FIRST so we're not stuck inside a folder we're about to delete.
# Without this, if you re-run the notebook, the shell's cwd is still inside the old
# repo folder — and once we rmtree it, every subsequent !shell command fails with
# 'shell-init: error retrieving current directory: getcwd' because the cwd no longer exists.
os.chdir('/content')

# Nuke any existing clone — Colab is ephemeral, no value in preserving state
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Fresh clone every time
!git clone {REPO_URL} {REPO_DIR}

# Create separate output directory for generated plots (NOT inside the repo)
os.makedirs(OUT_DIR, exist_ok=True)

# Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
# Add the repo's python/ to the import path
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'python'))

from solar_optimizer import (
    run_full_study, plot_all_dome_shapes, plot_hourly_harvest,
    plot_shape_comparison, plot_site_comparison, plot_3d_dome,
    hourly_harvest, panel_layout, compare_panels,
    SITES, SHAPES, PANELS, DEFAULT_PANEL,
)

print(f'Repo:         {REPO_DIR}')
print(f'Output dir:   {OUT_DIR}')
print(f'Default panel: {PANELS[DEFAULT_PANEL]["name"]}')
print(f'Available sites:  {list(SITES.keys())}')
print(f'Available shapes: {list(SHAPES.keys())}')
print(f'Available panels: {list(PANELS.keys())}')

## Compare all panel options at a glance

How many of each panel fit, peak power, coverage, and recommended series-pair topology.

In [ ]:
compare_panels(width_ft=7.5, length_ft=24, shape='low120')

## Full study — Petén with default panel (180W 24V)

Generates 5 plots: all vault shapes, hourly harvest, shape comparison, site comparison, 3D rendering. All saved to `/content/output/`.

In [ ]:
h = run_full_study(
    site='peten',
    width_ft=7.5, length_ft=24, floor_h_ft=4.0,
    panel=DEFAULT_PANEL,
    n_north=2,
    derate=0.87,
    save_dir=OUT_DIR,    # ← all plots land in /content/output/, NOT in the repo
    show=True,
)

## Try a different panel — 150W 12V for max coverage

Per the discussion in the chapter: 150W 12V in series quads gives ~25% more peak power and cheaper hail replacement. Output goes to a sub-folder so it doesn't overwrite the default run.

In [ ]:
out_150 = os.path.join(OUT_DIR, 'lensun_150_12')
os.makedirs(out_150, exist_ok=True)

h150 = run_full_study(
    site='peten',
    panel='lensun_150_12',
    save_dir=out_150,
    show=True,
)

## Sweep across all four sites — chapter spec panel

In [ ]:
for site in ['peten', 'kohala', 'austin', 'california']:
    site_dir = os.path.join(OUT_DIR, f'sites_{site}')
    os.makedirs(site_dir, exist_ok=True)
    print(f'\n--- {site} ---')
    run_full_study(site=site, save_dir=site_dir, show=False)

## Individual plots — explore one at a time

In [ ]:
# All 5 vault shapes
plot_all_dome_shapes(width_ft=7.5, floor_h_ft=4.0,
                     save_path=os.path.join(OUT_DIR, 'dome_shapes.png'))

In [ ]:
# Hourly harvest detail
fig, h = plot_hourly_harvest(
    site='peten', shape='low120', panel='lensun_180_24',
    save_path=os.path.join(OUT_DIR, 'hourly_default.png'),
)
print(f"Daily: {h['E_day']:.1f} kWh - Annual: {h['E_year']:.2f} MWh - Peak: {h['P_peak_arc']:.2f} kW")
print(f"Layout: {h['layout']['bus_topology']}")

In [ ]:
# 3D rendering of the chapter spec
plot_3d_dome(shape='low120', width_ft=7.5, floor_h_ft=4.0, length_ft=24,
             save_path=os.path.join(OUT_DIR, '3d_low120.png'))

## Download everything as a zip

In [ ]:
from google.colab import files

zip_path = shutil.make_archive('/content/azul_rocket_output', 'zip', OUT_DIR)
print(f'Created {zip_path}')
files.download(zip_path)

## Custom site — add your own location

In [ ]:
# Example: Atacama Desert, Chile
SITES['atacama'] = {
    'name':   'Atacama, Chile',
    'lat':    -24.0,
    'psh':    7.5,
    'albedo': 0.55,
    'canyon': 80,
}

atacama_dir = os.path.join(OUT_DIR, 'atacama')
os.makedirs(atacama_dir, exist_ok=True)
run_full_study(site='atacama', save_dir=atacama_dir, show=True)

## Browse the repo files (read-only)

The cloned repo lives at `/content/azul-rocket/`. Files there are read-only — don't edit them in Colab (you'd lose changes on the next run). To make changes, edit locally, commit, push to GitHub, then re-run this notebook.

In [ ]:
!ls -R {REPO_DIR} | head -40